# Distance and speed calculations with AISdb

AISdb turns raw AIS positions into physically meaningful features. This notebook
walks through the core geometry helpers in `aisdb.gis`, computing great-circle
distance and speed between consecutive positions of a real vessel track, then
adding a depth component for a submerged reference point. It closes with the
raster distance classes that measure how far each position sits from the nearest
shore, coast, and port.

## What you will learn

- Pull a real vessel track out of the bundled `example_data.db` with `DBQuery` and `TrackGen`
- Compute `delta_meters`, `delta_seconds`, and `delta_knots` along that track
- Measure 3D distance to a submerged point with `distance3D` and `vesseltrack_3D_dist`
- Attach `km_from_shore`, `km_from_coast`, and `km_from_port` using the raster distance classes

Companion GitBook pages: [Haversine Distance](https://aisviz.gitbook.io/documentation/tutorials/haversine-distance),
[Vessel Speed](https://aisviz.gitbook.io/documentation/tutorials/vessel-speed), and
[Coast, shore and ports](https://aisviz.gitbook.io/documentation/tutorials/coast-shore-and-ports).

In [ ]:
# %pip install aisdb
# On Google Colab, uncomment the line above and restart the runtime once it finishes.
# The raster distance section additionally uses only the standard library (os), which
# ships with Python, so no extra install is required here.

## Load a real vessel track

The GitBook pages use a hand-built five-point track so the numbers stay easy to
follow. Here we do the same thing with real data. `example_data.db` ships with
this repository and holds 40,000 dynamic AIS messages for 54 vessels in the Gulf
of Maine between 2022-01-01 and 2022-01-20. We open it with `SQLiteDBConn`, query
a one-week window with the `in_timerange_validmmsi` callback, and turn the rows
into track dictionaries with `TrackGen`.

Note that we only ever pass the connection object to `DBQuery`, never both a
connection and a path, because `gen_qry` asserts against that combination.

In [ ]:
import aisdb
from datetime import datetime
from aisdb import SQLiteDBConn, DBQuery, TrackGen
from aisdb.database import sqlfcn_callbacks

dbpath = './example_data.db'

with SQLiteDBConn(dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        callback=sqlfcn_callbacks.in_timerange_validmmsi,
        start=datetime(2022, 1, 1),
        end=datetime(2022, 1, 8),
    )
    # Materialise the tracks so we can reuse them across the cells below.
    tracks = list(TrackGen(qry.gen_qry(), decimate=False))

# Pick the longest track in the window so the distance and speed arrays are meaningful.
tracks_short = [max(tracks, key=lambda t: len(t['time']))]
track = tracks_short[0]
print(f"vessels in window: {len(tracks)}")
print(f"selected mmsi:     {track['mmsi']}")
print(f"positions:         {len(track['time'])}")

## Haversine distance

Two AIS positions are almost never on a flat plane. A vessel moving between
consecutive reports traces an arc across a curved Earth, and treating that arc as
a straight line introduces error that grows with distance. `aisdb.gis.delta_meters`
computes the great-circle (haversine) distance in meters between each pair of
consecutive positions in a track. It is the building block behind vessel speed and
the denoising encoder, which compares consecutive-position distances against a
threshold to flag noisy pings.

`delta_meters` returns one value fewer than the number of positions, because each
output is the distance between a position and the one that follows it.

In [ ]:
import numpy as np

dist_m = aisdb.gis.delta_meters(track)
print(f"legs: {len(dist_m)} (positions - 1 = {len(track['time']) - 1})")
print(f"total distance travelled: {dist_m.sum() / 1000:.1f} km")
print("first five leg distances (m):")
print(np.round(dist_m[:5].astype(float), 1))

## Vessel speed

Speed over ground follows directly from distance and elapsed time.
`aisdb.gis.delta_seconds` takes the difference between consecutive timestamps in
the track's `time` array, and `aisdb.gis.delta_knots` divides the haversine
distance by that interval and multiplies by 1.9438445 to convert meters per second
into knots.

$$Speed(knot) = \frac{Haversine\ Distance}{Time} \times 1.9438445$$

Internally `delta_knots` clamps each elapsed-time value to a minimum of one second
before dividing, so a duplicate or near-duplicate timestamp cannot produce a
division by zero or an unrealistically inflated speed. A speed that spikes far
above what the vessel type can sustain usually points to a corrupted or duplicated
position report rather than real movement, which is exactly the signal
`aisdb.denoising_encoder.remove_pings_wrt_speed` relies on.

In [ ]:
dt_s = aisdb.gis.delta_seconds(track)
speed_kn = aisdb.gis.delta_knots(track)

print("first five elapsed times (s):")
print(dt_s[:5])
print("first five speeds (knots):")
print(np.round(speed_kn[:5].astype(float), 2))
print(f"median speed: {np.median(speed_kn.astype(float)):.2f} knots")
print(f"max speed:    {speed_kn.max():.2f} knots")

## Distance to a submerged point

The haversine formula only accounts for horizontal distance across the Earth's
surface. When the reference point is not at sea level, such as a hydrophone, a
pipeline, or any other piece of subsea infrastructure, AISdb adds a vertical depth
component with `aisdb.gis.distance3D`. It combines the haversine distance with the
depth via the Pythagorean theorem, treating the depth as the third leg of a right
triangle.

$$Distance_{3D} = \sqrt{Haversine\ Distance^2 + Depth^2}$$

`aisdb.gis.vesseltrack_3D_dist` applies this across an entire track, appending the
result to each position under a new dynamic key (`distance_metres` by default).
Unlike `delta_meters`, it returns one value per position rather than one per leg,
since each distance is measured from the fixed subsea point rather than between
consecutive positions.

In [ ]:
# A single point-to-point 3D distance: two surface positions, reference 150 m deep.
d3d = aisdb.gis.distance3D(
    x1=track['lon'][0], y1=track['lat'][0],
    x2=track['lon'][1], y2=track['lat'][1],
    depth_metres=150,
)
print(f"3D distance between the first two positions (150 m depth): {d3d:.1f} m")

# Across the whole track, measured to a fixed subsea point in the Gulf of Maine.
x1, y1, z1 = -68.0, 43.0, 150
tracks_3d = aisdb.gis.vesseltrack_3D_dist(tracks_short, x1=x1, y1=y1, z1=z1)

for t in tracks_3d:
    d = t['distance_metres']
    print(f"values: {len(d)} (one per position = {len(t['time'])})")
    print("first five distances to the subsea point (m):")
    print(np.round(d[:5].astype(float), 1))

## Distance from shore, coast, and port

`ShoreDist`, `CoastDist`, and `PortDist` all follow the same pattern. Each one
downloads its raster from the AISdb data server into `data_dir`, merges the raster
against the provided track list, and stores the result under a new dynamic key
(`km_from_shore`, `km_from_coast`, `km_from_port`).

Two operational details matter for running this unattended. The helpers assert
that `data_dir` already exists, so we create it with `os.makedirs(..., exist_ok=True)`
first. And they cache the downloaded archive, so pointing every class at one
persistent `./raster_data/` directory means a re-run skips the download entirely.

Be honest about the download sizes before running these cells. The shore raster is
about 40 MB and the coast raster about 59 MB, both small enough to fetch on demand.
The port raster is roughly 1.3 GB, so it lives in an optional cell further down,
guarded by a flag.

In [ ]:
import os

data_dir = './raster_data/'
os.makedirs(data_dir, exist_ok=True)
print(f"raster cache directory ready: {os.path.abspath(data_dir)}")

### Distance from shore

`aisdb.webdata.shore_dist.ShoreDist` calculates the nearest distance to shore.
Calling `get_distance` downloads the shore raster (about 40 MB) on first use,
merges it against the track list, and adds a `km_from_shore` key to both
`track.keys()` and the `track['dynamic']` set.

In [ ]:
from aisdb.webdata.shore_dist import ShoreDist

with ShoreDist(data_dir=data_dir) as sdist:
    for t in sdist.get_distance(tracks_short):
        assert 'km_from_shore' in t.keys()
        assert 'km_from_shore' in t['dynamic']
        print("first five km_from_shore values:")
        print(t['km_from_shore'][:5])

### Distance from coast

`CoastDist` works the same way against the coastline raster (about 59 MB),
storing its result under `km_from_coast`.

In [ ]:
from aisdb.webdata.shore_dist import CoastDist

with CoastDist(data_dir=data_dir) as cdist:
    for t in cdist.get_distance(tracks_short):
        assert 'km_from_coast' in t.keys()
        assert 'km_from_coast' in t['dynamic']
        print("first five km_from_coast values:")
        print(t['km_from_coast'][:5])

### Distance from the nearest port (optional, heavy download)

`PortDist` follows the identical pattern and stores its result under
`km_from_port`, but its raster archive is roughly 1.3 GB. To keep the default,
headless run of this notebook light, the cell below is guarded by `RUN_PORTDIST`,
which defaults to `False`. Flip it to `True` locally when you have the bandwidth
and disk space, and the download will be cached in `./raster_data/` for future
runs.

In [ ]:
RUN_PORTDIST = False  # set to True locally to fetch the ~1.3 GB port raster

if RUN_PORTDIST:
    from aisdb.webdata.shore_dist import PortDist

    with PortDist(data_dir=data_dir) as pdist:
        for t in pdist.get_distance(tracks_short):
            assert 'km_from_port' in t.keys()
            assert 'km_from_port' in t['dynamic']
            print("first five km_from_port values:")
            print(t['km_from_port'][:5])
else:
    print("PortDist skipped. Set RUN_PORTDIST = True to download the ~1.3 GB port raster.")

## Next steps

These distance and speed features feed directly into cleaning and interpolation.
Continue with the [track interpolation](https://aisviz.gitbook.io/documentation/tutorials/track-interpolation)
tutorial to resample tracks onto a regular time grid, or revisit the
[data cleaning](https://aisviz.gitbook.io/documentation/tutorials/data-cleaning)
page to see how the speed signal drives the denoising encoder.

For the fuller narrative behind each helper, see the companion GitBook pages:
[Haversine Distance](https://aisviz.gitbook.io/documentation/tutorials/haversine-distance),
[Vessel Speed](https://aisviz.gitbook.io/documentation/tutorials/vessel-speed), and
[Coast, shore and ports](https://aisviz.gitbook.io/documentation/tutorials/coast-shore-and-ports).